# Data management with DigitalHub

Assets are stored in repositories and registered in the catalog. 

This tutorial demonstrates how to create, update, download assets from inside and outside the platform.


## Project Initialization

Initialize our DigitalHub project. This creates a space where we can manage our data items, functions, workflows, and deployments.

We'll use the `username` as param to create a personal project in the shared space.


In [ ]:
import digitalhub as dh
import getpass as gt

USERNAME = gt.getuser()
project = dh.get_or_create_project(f"{USERNAME}-data-ops")
print(project.name)

## Data management

We can store and retrieve artifacts, data items and models via sdk, picking from local or remote sources.


In [ ]:
# generate a random dataset
import pandas as pd
import numpy as np  
df =  pd.DataFrame(np.random.rand(100, 4), columns=list('ABCD'))
df.to_csv("dataset.csv", index=False)

In [ ]:
# upload and register the dataset
di = project.log_table(name="dataset", source="dataset.csv")

In [ ]:
# we can also directly upload the dataset without saving it locally
di2 = project.log_table(name="dataframe", data=df)

Let's try with a remote resource.

In [ ]:
## remote 
URL = "https://opendata.comune.bologna.it/api/explore/v2.1/catalog/datasets/rilevazione-flusso-veicoli-tramite-spire-anno-2023/exports/csv?limit=50000&lang=it&timezone=Europe%2FRome&use_labels=true&delimiter=%3B"
filename = "rilevazione-flusso-veicoli-tramite-spire-anno-2023.csv"

In [ ]:

di3 = project.register_table(name=filename, source=URL)

In [ ]:
di3

What about reading it back? 
We can get a file, either as full download or as temporary file.

In [ ]:
path = di.download(overwrite=True)

In [ ]:
path

Can we read it back?

In [ ]:
pdf = pd.read_csv(path, sep=";")

In [ ]:
pdf.head()

We can also directly load it as dataframe, without downloading a local copy.

In [ ]:
df = di.as_df(sep=";")

In [ ]:
df.head()

Maybe a parquet is more suitable?

In [ ]:
pq_file = "dataitem/dataset.parquet"
di.as_df().to_parquet(pq_file)

In [ ]:
pq_file

## Functions and dataitems

The SDK can handle complex return types and directly manage the creation of resources via `decorators`.

https://scc-digitalhub.github.io/sdk-docs/reference/runtimes/python/define-function/#handler-and-outputs

Define a function which creates a dataframe as result. Pick from cell #2
```
# generate a random dataset
import pandas as pd
import numpy as np  
df =  pd.DataFrame(np.random.rand(100, 4), columns=list('ABCD'))
df.to_csv("dataset.csv", index=False)
```


In [ ]:
%%writefile "generate.py"

from digitalhub_runtime_python import handler
import pandas as pd
import numpy as np  

@handler(outputs=["data"])
def generate():
    # produce a Dataitem

   



In [ ]:
func = project.new_function(name="generate-data",
                            kind="python",
                            python_version="PYTHON3_12",
                            code_src="generate.py",
                            handler="generate")

In [ ]:
run = func.run("job", wait=True, local_execution=True)


In [ ]:
run.outputs()

The output is a valid dataitem. 

In [ ]:
run.outputs()['data']

We can use it as such, for example by downloading or reading as new dataframe the table.

In [ ]:
run.outputs()['data'].as_df().head()